# 주술회전 (呪術廻戦) - 주술 배틀 게임

주술사를 선택하고 저주와 싸우는 간단한 턴제 배틀 게임입니다.

In [ ]:
import random
import time


CHARACTERS = {
    "1": {
        "name": "이타도리 유지",
        "hp": 120,
        "attack": 25,
        "defense": 10,
        "skills": {
            "축권": {"damage": 45, "cost": 20, "desc": "주술을 담은 강력한 주먹!"},
            "흑섬": {"damage": 70, "cost": 40, "desc": "연속 타격의 폭풍!"},
        },
        "max_mp": 100,
    },
    "2": {
        "name": "고조 사토루",
        "hp": 100,
        "attack": 30,
        "defense": 15,
        "skills": {
            "무량공처": {"damage": 90, "cost": 50, "desc": "영역전개! 무한의 공간!"},
            "허식 자주": {"damage": 55, "cost": 25, "desc": "수렴과 발산의 힘!"},
        },
        "max_mp": 120,
    },
    "3": {
        "name": "후시구로 메구미",
        "hp": 100,
        "attack": 22,
        "defense": 12,
        "skills": {
            "옥견 혼": {"damage": 40, "cost": 15, "desc": "식신 쌍흑견 소환!"},
            "기은도해": {"damage": 80, "cost": 45, "desc": "영역전개! 그림자의 바다!"},
        },
        "max_mp": 100,
    },
    "4": {
        "name": "오코츠 유타",
        "hp": 110,
        "attack": 28,
        "defense": 13,
        "skills": {
            "리카 소환": {"damage": 60, "cost": 30, "desc": "특급 과주 리카를 소환!"},
            "술식 모사": {"damage": 50, "cost": 20, "desc": "상대의 술식을 복사!"},
        },
        "max_mp": 130,
    },
}

CURSES = [
    {
        "name": "특급 저주령 - 조고",
        "hp": 150,
        "attack": 22,
        "defense": 8,
        "skills": ["영혼의 변형", "흑점선", "자기수복"],
    },
    {
        "name": "특급 저주령 - 하나미",
        "hp": 130,
        "attack": 20,
        "defense": 12,
        "skills": ["초목조례", "화휘", "뿌리 공격"],
    },
    {
        "name": "저주의 왕 - 스쿠나 (손가락 3개)",
        "hp": 180,
        "attack": 28,
        "defense": 10,
        "skills": ["해체", "봉인", "개"],
    },
    {
        "name": "저주의 왕 - 스쿠나 (손가락 15개)",
        "hp": 300,
        "attack": 40,
        "defense": 15,
        "skills": ["해체", "봉인", "복마전신", "영역전개: 복마전신"],
    },
]


def hp_bar(current, maximum, length=20):
    filled = int(length * current / maximum)
    bar = "█" * filled + "░" * (length - filled)
    return f"[{bar}] {current}/{maximum}"


def display_status(player_name, player_hp, player_max_hp, player_mp, player_max_mp,
                   enemy_name, enemy_hp, enemy_max_hp):
    print("\n" + "=" * 50)
    print(f"  🧑 {player_name}")
    print(f"  HP: {hp_bar(player_hp, player_max_hp)}")
    print(f"  MP: {hp_bar(player_mp, player_max_mp)}")
    print(f"\n  👹 {enemy_name}")
    print(f"  HP: {hp_bar(enemy_hp, enemy_max_hp)}")
    print("=" * 50)


def battle(player, enemy):
    p_hp = player["hp"]
    p_max_hp = player["hp"]
    p_mp = player["max_mp"]
    p_max_mp = player["max_mp"]
    e_hp = enemy["hp"]
    e_max_hp = enemy["hp"]
    turn = 1

    print(f"\n⚔️  {player['name']} vs {enemy['name']}  ⚔️")
    print("전투 시작!\n")

    while p_hp > 0 and e_hp > 0:
        display_status(player["name"], p_hp, p_max_hp, p_mp, p_max_mp,
                       enemy["name"], e_hp, e_max_hp)

        print(f"\n--- 턴 {turn} ---")
        print("행동을 선택하세요:")
        print("  1. 일반 공격")
        skill_list = list(player["skills"].items())
        for i, (name, info) in enumerate(skill_list):
            print(f"  {i+2}. {name} (MP:{info['cost']}) - {info['desc']}")
        print(f"  {len(skill_list)+2}. 주력 회복 (MP +30)")

        choice = input("\n> ").strip()

        # 플레이어 턴
        damage = 0
        if choice == "1":
            damage = player["attack"] + random.randint(-5, 10)
            damage = max(0, damage - enemy["defense"])
            print(f"\n👊 {player['name']}의 일반 공격! {damage} 데미지!")
        elif choice == str(len(skill_list) + 2):
            recover = min(30, p_max_mp - p_mp)
            p_mp += recover
            print(f"\n🔵 주력을 집중합니다... MP +{recover} 회복!")
        else:
            skill_idx = int(choice) - 2 if choice.isdigit() else 0
            if 0 <= skill_idx < len(skill_list):
                skill_name, skill_info = skill_list[skill_idx]
                if p_mp >= skill_info["cost"]:
                    p_mp -= skill_info["cost"]
                    damage = skill_info["damage"] + random.randint(-5, 15)
                    damage = max(0, damage - enemy["defense"])
                    crit = random.random() < 0.15
                    if crit:
                        damage = int(damage * 1.8)
                        print(f"\n💥 크리티컬! {skill_name}!! {damage} 데미지!")
                    else:
                        print(f"\n✨ {skill_name}! {damage} 데미지!")
                else:
                    print("\n⚠️ MP가 부족합니다! 일반 공격으로 전환!")
                    damage = player["attack"] + random.randint(-5, 5)
                    damage = max(0, damage - enemy["defense"])
            else:
                print("\n잘못된 선택! 일반 공격으로 전환!")
                damage = player["attack"] + random.randint(-5, 5)
                damage = max(0, damage - enemy["defense"])

        e_hp -= damage
        e_hp = max(0, e_hp)

        if e_hp <= 0:
            print(f"\n🎉 {enemy['name']}을(를) 쓰러뜨렸다!")
            print(f"🏆 {player['name']} 승리! 주살 완료!")
            break

        # 적 턴
        e_skill = random.choice(enemy["skills"])
        e_damage = enemy["attack"] + random.randint(-3, 15)
        e_damage = max(0, e_damage - player["defense"])
        crit = random.random() < 0.1
        if crit:
            e_damage = int(e_damage * 1.5)
            print(f"\n💀 {enemy['name']}의 {e_skill}! 강력한 일격! {e_damage} 데미지!")
        else:
            print(f"\n👹 {enemy['name']}의 {e_skill}! {e_damage} 데미지!")

        p_hp -= e_damage
        p_hp = max(0, p_hp)

        if p_hp <= 0:
            print(f"\n💀 {player['name']}이(가) 쓰러졌다...")
            print("😈 패배... 저주에 잠식당했습니다.")
            break

        # MP 자연 회복
        p_mp = min(p_max_mp, p_mp + 5)
        turn += 1

    display_status(player["name"], p_hp, p_max_hp, p_mp, p_max_mp,
                   enemy["name"], e_hp, e_max_hp)
    print("\n전투 종료!")


def main():
    print("╔══════════════════════════════════════╗")
    print("║    주술회전 - 주술 배틀 게임         ║")
    print("║    呪術廻戦 - JUJUTSU BATTLE         ║")
    print("╚══════════════════════════════════════╝")
    print()
    print("주술사를 선택하세요:")
    for key, char in CHARACTERS.items():
        print(f"  {key}. {char['name']} (HP:{char['hp']} ATK:{char['attack']} DEF:{char['defense']})")

    choice = input("\n> ").strip()
    if choice not in CHARACTERS:
        choice = "1"
    player = CHARACTERS[choice]
    print(f"\n✅ {player['name']} 선택!")

    print("\n난이도를 선택하세요:")
    print("  1. 4급 임무 (쉬움)")
    print("  2. 2급 임무 (보통)")
    print("  3. 특급 임무 (어려움)")
    print("  4. 스쿠나전 (지옥)")

    diff = input("\n> ").strip()
    diff_idx = int(diff) - 1 if diff.isdigit() and 1 <= int(diff) <= 4 else 0
    enemy = CURSES[diff_idx].copy()
    print(f"\n⚠️ {enemy['name']} 출현!")

    battle(player, enemy)

    again = input("\n다시 하시겠습니까? (y/n): ").strip().lower()
    if again == "y":
        main()


main()